In [ ]:
import sys
!{sys.executable} -m pip install pandas numpy matplotlib scipy yfinance pandas_datareader
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm
from scipy.optimize import brentq
import yfinance as yf
from datetime import datetime

In [ ]:
def bs_call(s, k, t, r, sigma):
    d1=(np.log(s/k)+(r+0.5*sigma**2)*t)/(sigma*np.sqrt(t))
    d2=d1-sigma*np.sqrt(t)
    return s*norm.cdf(d1)-k*np.exp((-1)*r*t)*norm.cdf(d2)

def bs_putt(s, k, t, r, sigma):
    d1=(np.log(s/k)+(r+0.5*sigma**2)*t)/(sigma*np.sqrt(t))
    d2=d1-sigma*np.sqrt(t)
    return k*np.exp((-1)*r*t)*norm.cdf((-1)*d2)-s*norm.cdf((-1)*d1)

In [ ]:
def bs_call_greeks(s, k, t, r, sigma):
    d1 = (np.log(s/k) + (r + 0.5*sigma**2)*t) / (sigma*np.sqrt(t))
    d2 = d1 - sigma*np.sqrt(t)
    phi = norm.pdf(d1)
    N1, N2 = norm.cdf(d1), norm.cdf(d2)
    
    price = s*N1 - k*np.exp(-r*t)*N2
    delta = N1
    gamma = phi / (s*sigma*np.sqrt(t))
    vega  = s*phi*np.sqrt(s)
    theta = -s*phi*sigma/(2*np.sqrt(t)) - r*k*np.exp(-r*t)*N2
    rho   = k*t*np.exp(-r*t)*N2
    
    return {
        'price': price, 'delta': delta, 'gamma': gamma,
        'vega': vega, 'theta': theta, 'rho': rho,
        'vega_per_1pct': vega/100,
        'theta_per_day': theta/365,
        'rho_per_1pct':  rho/100,
    }

g = bs_call_greeks(100, 100, 1.0, 0.05, 0.2)
for k, v in g.items():
    print(f"{k:>15}: {v:10.6f}")

In [ ]:
def finite_diff_check(s, k, t, r, sigma, h=1e-4):
    analytic = bs_call_greeks(s, k, t, r, sigma)
    
    delta_fd = (bs_call(s+h, k, t, r, sigma) - bs_call(s-h, k, t, r, sigma)) / (2*h)
    gamma_fd = (bs_call(s+h, k, t, r, sigma) - 2*bs_call(s, k, t, r, sigma) 
                + bs_call(s-h, k, t, r, sigma)) / h**2
    vega_fd  = (bs_call(s, k, t, r, sigma+h) - bs_call(s, k, t, r, sigma-h)) / (2*h)
    theta_fd = (bs_call(s, k, t+h, r, sigma) - bs_call(s, k, t-h, r, sigma)) / (2*h)
    rho_fd   = (bs_call(s, k, t, r+h, sigma) - bs_call(s, k, t, r-h, sigma)) / (2*h)
    
    print(f"{'Greek':>8} {'Analytic':>12} {'Finite diff':>14} {'|diff|':>12}")
    for name, a, fd in [('delta', analytic['delta'], delta_fd),
                        ('gamma', analytic['gamma'], gamma_fd),
                        ('vega',  analytic['vega'],  vega_fd),
                        ('theta', analytic['theta'], theta_fd),
                        ('rho',   analytic['rho'],   rho_fd)]:
        print(f"{name:>8} {a:12.6f} {fd:14.6f} {abs(a-fd):12.2e}")

finite_diff_check(100, 100, 1.0, 0.05, 0.2)

In [ ]:
ticker="SPY"
tk=yf.Ticker(ticker)
hist=yf.download(ticker, period="1y", auto_adjust=True)["Close"]
s0=float(hist.iloc[-1].item())
print(f"Ціна{ticker}:${s0:.2f}")
logret=np.log(hist / hist.shift(1)).dropna()
sigma_hist=float(logret.std().item()*np.sqrt(252))
print(f"Історична σ:{sigma_hist*100:.2f}%")


In [ ]:
expiries=tk.options
print("Доступні експірації:", expiries[:10])
expiry=expiries[8]
t=(pd.Timestamp(expiry)-pd.Timestamp.today()).days/365
print(f"Обрана експірація:{expiry}, t={T:.4f} років")
chain=tk.option_chain(expiry)
calls=chain.calls.copy()
calls['mid']=(calls['bid']+calls['ask'])/2
calls=calls[calls['mid']>0]
calls[['strike', 'bid', 'ask', 'mid', 'volume', 'impliedVolatility']].head(10)

In [ ]:
try:
    from pandas_datareader import data as pdr
    r_series=pdr.DataReader("DGS3MO", "fred", start=pd.Timestamp.today()-pd.Timedelta(days=30))
    r=float(r_series.dropna().iloc[-1].item())/100
except Exception as e:
    print(f"FRED недоступний ({e}), використовуємо дефолт")
    r=0.045
print(f"r={r*100:.3f}%")

In [ ]:
results = []
for _, row in calls.iterrows():
    k = row['strike']
    bs_price = bs_call(s0, k, t, r, sigma_hist)
    results.append({
        'strike': k,
        'moneyness': k/s0,
        'market_mid': row['mid'],
        'bs_price': bs_price,
        'diff': bs_price - row['mid'],
        'market_iv': row['impliedVolatility'],
    })

comp = pd.DataFrame(results)
comp = comp[(comp['moneyness'] > 0.9) & (comp['moneyness'] < 1.1)]
comp.round(4)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 9))

s_range = np.linspace(50, 150, 200)
k_, t_, r_, s_ = 100, 1.0, 0.05, 0.2


axes[0,0].plot(s_range, [bs_call(s, k_, t_, r_, s_) for s in s_range], lw=2)
axes[0,0].plot(s_range, np.maximum(s_range - k_*np.exp(-r_*t_), 0), 
               '--', alpha=0.5, label='intrinsic')
axes[0,0].axvline(k_, color='gray', ls=':', label=f'k={k_}')
axes[0,0].set_xlabel('s'); axes[0,0].set_ylabel('Call price')
axes[0,0].set_title('C(s)'); axes[0,0].legend()


deltas = [bs_call_greeks(S, K_, T_, r_, s_)['delta'] for s in s_range]
gammas = [bs_call_greeks(S, K_, T_, r_, s_)['gamma'] for s in s_range]
ax2 = axes[0,1]; ax2b = ax2.twinx()
ax2.plot(s_range, deltas, 'b-', lw=2, label='delta')
ax2b.plot(s_range, gammas, 'r-', lw=2, label='gamma')
ax2.set_xlabel('s'); ax2.set_ylabel('delta', color='b')
ax2b.set_ylabel('gamma', color='r')
ax2.axvline(K_, color='gray', ls=':')
ax2.set_title('delta & gamma')


vegas  = [bs_call_greeks(s, k_, t_, r_, s_)['vega']  for s in s_range]
thetas = [bs_call_greeks(s, k_, t_, r_, s_)['theta'] for s in s_range]
ax3 = axes[1,0]; ax3b = ax3.twinx()
ax3.plot(s_range, vegas, 'g-', lw=2, label='Vega')
ax3b.plot(s_range, thetas, 'm-', lw=2, label='Theta')
ax3.set_xlabel('s'); ax3.set_ylabel('vega', color='g')
ax3b.set_ylabel('theta', color='m')
ax3.axvline(K_, color='gray', ls=':')
ax3.set_title('vega & theta')

axes[1,1].plot(comp['moneyness'], comp['iv_computed']*100, 'o-', label='Implied vol')
axes[1,1].axhline(sigma_hist*100, color='red', ls='--', label=f'σ hist = {sigma_hist*100:.1f}%')
axes[1,1].set_xlabel('Moneyness k/s'); axes[1,1].set_ylabel('IV, %')
axes[1,1].set_title(f'Volatility smile ({ticker}, exp {expiry})')
axes[1,1].legend()

plt.tight_layout()
plt.show()